## Section C: Safety Incident Database

In [2]:
import pandas as pd
from pathlib import Path

FILE_PATH = Path("MINN2020A_Complete_Realistic_Project_Data_CLEANED.xlsx")


def save_sheet(data, sheet_name):
    with pd.ExcelWriter(
        FILE_PATH,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace") as writer:
        data.to_excel(writer, sheet_name=sheet_name, index=False)

In [3]:
INCIDENT_COLUMNS = [
    "Incident ID",
    "Date",
    "Time",
    "Shift",
    "Location",
    "Department",
    "Worker ID",
    "Incident Type",
    "Severity",
    "Injury Status",
    "Lost Time Injury",
    "Cause",
    "Corrective Action",
    "Incident Status"
]


def load_incidents():
    return pd.read_excel(
        FILE_PATH,
        sheet_name="Safety Incidents"
    )


def add_incident(details):
    incidents = load_incidents()

    # Check that all fields except the ID have been supplied
    required_fields = INCIDENT_COLUMNS[1:]
    missing_fields = [
        field for field in required_fields
        if field not in details
    ]

    if missing_fields:
        raise ValueError(
            f"Missing fields: {missing_fields}"
        )

    # Generate the next ID using the existing INC-0001 format
    numbers = pd.to_numeric(
        incidents["Incident ID"]
        .astype(str)
        .str.extract(r"(\d+)$")[0],
        errors="coerce"
    )

    next_number = int(numbers.max()) + 1 if numbers.notna().any() else 1
    incident_id = f"INC-{next_number:04d}"

    details = details.copy()
    details["Incident ID"] = incident_id

    new_record = pd.DataFrame(
        [details],
        columns=INCIDENT_COLUMNS
    )

    incidents = pd.concat(
        [incidents, new_record],
        ignore_index=True
    )

    save_sheet(incidents, "Safety Incidents")

    return incident_id


def view_incidents():
    return load_incidents()


def search_incident(incident_id):
    incidents = load_incidents()

    result = incidents[
        incidents["Incident ID"].astype(str).str.upper()
        == incident_id.strip().upper()
    ]

    return result


def filter_incidents(column, value):
    incidents = load_incidents()

    if column not in INCIDENT_COLUMNS:
        raise ValueError("Invalid filter column")

    return incidents[
        incidents[column].astype(str).str.contains(
            str(value),
            case=False,
            na=False
        )
    ]


def count_incidents(column="Incident Type"):
    incidents = load_incidents()

    if column not in ["Incident Type", "Severity"]:
        raise ValueError(
            "Choose Incident Type or Severity"
        )

    return incidents[column].value_counts()


def incident_trends():
    incidents = load_incidents()

    incidents["Date"] = pd.to_datetime(
        incidents["Date"],
        errors="coerce"
    )

    incidents = incidents.dropna(subset=["Date"])

    incidents["Month"] = (
        incidents["Date"].dt.to_period("M")
    )

    return incidents.groupby("Month").size()

## Section D: Equipment database

In [4]:
EQUIPMENT_COLUMNS = [
    "Equipment ID",
    "Equipment Type",
    "Manufacturer",
    "Department",
    "Operating Hours (h)",
    "Temperature (°C)",
    "Vibration (mm/s)",
    "Fuel Consumption (L/h)",
    "Brake Status",
    "Tyre Status",
    "Engine Status",
    "Maintenance Status",
    "Downtime (h)",
    "Availability (%)"
]


def load_equipment():
    return pd.read_excel(
        FILE_PATH,
        sheet_name="Equipment"
    )


def add_equipment(details):
    equipment = load_equipment()

    # Check that all fields except the ID have been supplied
    required_fields = EQUIPMENT_COLUMNS[1:]
    missing_fields = [
        field for field in required_fields
        if field not in details
    ]

    if missing_fields:
        raise ValueError(
            f"Missing fields: {missing_fields}"
        )

    # Generate the next ID using the existing EQ-001 format
    numbers = pd.to_numeric(
        equipment["Equipment ID"]
        .astype(str)
        .str.extract(r"(\d+)$")[0],
        errors="coerce"
    )

    next_number = int(numbers.max()) + 1 if numbers.notna().any() else 1
    equipment_id = f"EQ-{next_number:03d}"

    details = details.copy()
    details["Equipment ID"] = equipment_id

    new_record = pd.DataFrame(
        [details],
        columns=EQUIPMENT_COLUMNS
    )

    equipment = pd.concat(
        [equipment, new_record],
        ignore_index=True
    )

    save_sheet(equipment, "Equipment")

    return equipment_id


def view_equipment():
    return load_equipment()


def search_equipment(equipment_id):
    equipment = load_equipment()

    result = equipment[
        equipment["Equipment ID"].astype(str).str.upper()
        == equipment_id.strip().upper()
    ]

    return result


def filter_equipment(column, value):
    equipment = load_equipment()

    if column not in EQUIPMENT_COLUMNS:
        raise ValueError("Invalid filter column")

    return equipment[
        equipment[column].astype(str).str.contains(
            str(value),
            case=False,
            na=False
        )
    ]